# Bayesian Compositional Timeseries Model w/ PyMC

In [32]:
import pymc as pm
import numpy as np
import pandas as pd
import arviz as az
import pytensor.tensor as pt
import matplotlib.pyplot as plt
from pygam import LinearGAM, s
from datetime import datetime, timedelta
import os

In [11]:
data_path = 'modeldat.csv'
data_dir = os.path.dirname(data_path)
project_root = os.path.dirname(data_dir)

### Construct X and y Data:

In [14]:
n_test = 10 # forecast horizon
n_data_lag = 3
training_end_date = datetime(2025, 8, 15)
last_training_data_date = training_end_date - timedelta(days=n_data_lag)

df = pd.read_csv(data_path)

## Construct outcome variable, time index, day index
y_df = df[['operational_day']]
y_df = y_df.rename(columns={'operational_day': 'dt'})
y_df['dt'] = pd.to_datetime(y_df['dt'])
y_df['t'] = np.arange(len(y_df))
y_df['day'] = y_df['dt'].dt.day_of_week
y_df['y_obs'] = df['estimated_avoidable_deaths']
y_df['log_y_obs'] = np.log1p(y_df['y_obs'])

# Use z-score for covariates
X_raw = df.drop(columns=['operational_day', 'estimated_avoidable_deaths']).replace(-9999, np.nan)
X_cleaned = X_raw.fillna(X_raw.median())
X_scaled = (X_cleaned - X_cleaned.mean()) / X_cleaned.std()

# convert to np.ndarray of shape (n_timesteps, n_lags, n_covariates)
lag_list = []
for i in range(n_test+n_data_lag+1):
    # Shift the data down by i days 
    shifted_X = X_scaled.shift(i).add_suffix(f'_lag{i}')
    lag_list.append(shifted_X)
X_final = np.stack([df.values for df in lag_list], axis=1)

index = pd.MultiIndex.from_product(
    [y_df['t'], -np.arange(n_test+n_data_lag+1), X_raw.columns],
    names=["t", "lag", "covariate_name"]
)
X_df = pd.DataFrame({"covariate_value": X_final.ravel()}, index=index)

# Drop the first n_test + n_data_lag timesteps in both dataframes
bad_times = X_df["covariate_value"].isna().groupby(level="t").any() # detect times containing at least one NaN
bad_times = bad_times[bad_times].index # Keep only times where NaNs were found
X_df = X_df.drop(index=bad_times, level="t")
X_df = X_df.reset_index()
y_df = y_df[~y_df['t'].isin(bad_times.values)]
X_df['dt'] = X_df['t'].map(dict(zip(y_df['t'], y_df['dt'])))

# Estimate seasonal amplitude directly from the data
mu0_est = np.median(y_df['log_y_obs'])
A_est = np.quantile(y_df['log_y_obs'], q=0.85) - np.median(y_df['log_y_obs'])

### Estimate Variance in the Data:

In [27]:
# Fit GAM to obtain a smooth trend
gam = LinearGAM(s(0), fit_intercept=True).fit(y_df['t'], y_df['log_y_obs'])
gam_trend = gam.predict(y_df['t'])

# Compute standard deviation of residuals
gam_residuals = y_df['log_y_obs'] - gam_trend
sigma_est = np.std(gam_residuals, ddof=1)
print(f"Estimated residual SD: {sigma_est:.4f}")

Estimated residual SD: 0.1493


### Extract Training Data:

In [10]:
# Split the dataframes using dates
y_df_train = y_df[y_df['dt'] <= last_training_data_date]
X_df_train = X_df[X_df['dt'] <= last_training_data_date] 

# Convert to numpy arrays for the pyMC model
t_train = y_df_train['t'].values
dt_train = y_df_train['dt'].values
d_train = y_df_train['day'].values
y_train = y_df_train['log_y_obs'].values
X_train = X_df_train['covariate_value'].values.reshape(len(X_df_train['t'].unique()), len(X_df_train['lag'].unique()), len(X_df_train['covariate_name'].unique()))

### Build PyMC Model:

In [12]:
lags_lasso = np.abs(X_df['lag'].unique())

# Coordinates
coords = { 
    "date": dt_train,
    "day": ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'],
    "covariates": X_df['covariate_name'].unique(),
    "lags": X_df['lag'].unique()
        }

with pm.Model(coords=coords) as nhs_model:

    # Data containers for swapping
    t_shared = pm.Data("t_shared", t_train, dims="date")
    d_shared = pm.Data("d_shared", d_train, dims="date")
    X_shared = pm.Data("X_shared", X_train, dims=("date", "lags", "covariates"))
    y_shared = pm.Data("y_shared", y_train, dims="date")

    # Long-term seasonal model
    mu0 = pm.Normal("mu0", mu=mu0_est, sigma=1/3)
    A = pm.LogNormal("A", mu=A_est, sigma=1/3)
    phi = 5.2
    mu_seasonal = pm.Deterministic("mu_seasonal", mu0 + A * pm.math.cos((2 * np.pi * t_shared / 365.25) - phi), dims="date")

    # Bayesian LASSO with distributed lags
    lam = 0.1
    lag_weights = np.exp(-lam * lags_lasso)
    betas = pm.Laplace("betas", mu=0, b=lag_weights[::-1,None]*(0.015/3)/(np.sqrt(2)), dims=("lags", "covariates")) # normal var = sigma^2 and Laplace var = 2b^2
    mu_covariates = pm.Deterministic("mu_covariates", pm.math.sum(X_shared * betas, axis=(1,2)), dims="date")

    # Week-day random effect
    phi_d = pm.Normal("phi_d", mu=0, sigma=0.05/3, dims="day") 
    day_of_week_effect = phi_d[d_shared]

    # AR(1) memory
    rho_AR = 0.5 
    sigma_AR = pm.HalfNormal("sigma_AR", sigma=0.01/3)
    start_AR = (y_shared - (mu_seasonal + mu_covariates + day_of_week_effect))[0]
    mu_AR = pm.AR(
        "mu_AR", 
        rho=[rho_AR], 
        sigma=sigma_AR, 
        init_dist=pm.Normal.dist(start_AR, np.sqrt(sigma_AR**2/(1-rho_AR**2))),
        dims="date" 
    )
    
    # Likelihood
    mu_total = pm.Deterministic("mu_total", mu_seasonal + day_of_week_effect +  mu_covariates + mu_AR, dims="date")
    sigma_obs = pm.HalfNormal("sigma_obs", sigma=0.15/3) # 0.15 = estimated residual standard deviation (divide by 3 --> 95% of density between -0.15 to +0.15)
    obs = pm.Normal("obs", mu=mu_total, sigma=sigma_obs, observed=y_shared, dims="date")

### Train Model:

In [15]:
with nhs_model:
    trace = pm.sample(draws=250, tune=250, chains=4, progressbar=True, target_accept=0.9,
                      init='adapt_diag',                                                              
                      initvals=4*[{'mu0': mu0_est, 'A': A_est, 'sigma_AR': 0.01, 'sigma_obs': 0.15},])  
    train = pm.sample_posterior_predictive(trace)

Initializing NUTS using adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [mu0, A, betas, phi_d, sigma_AR, mu_AR, sigma_obs]


Output()

Sampling 4 chains for 250 tune and 250 draw iterations (1_000 + 1_000 draws total) took 317 seconds.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details
Sampling: [obs]


Output()

### Plotting and Evaluation Functions for a Set of Forecast Start Dates With Average MSE

In [97]:
def plot_window_trajectory(train, forecast, y_df_test, last_training_data_date, dt_test, anchor_date, null_forecast, null_score=None, model_score=None):
    save_dir = os.path.join(project_root, "figures", "forecasts")
    os.makedirs(save_dir, exist_ok=True)
    
    show_n_tr = 60
    dates_train = train.posterior_predictive.coords['date'].values
    dates_test = forecast.posterior_predictive.coords['date'].values

    plt.figure(figsize=(14, 6))

    # Historical Training Data & Fit
    plt.plot(dates_train[-show_n_tr:], np.expm1(train.observed_data['obs'].values[-show_n_tr:]), color="black", linewidth=2.5)
    plt.plot(dates_train[-show_n_tr:], np.expm1(train.posterior_predictive['obs'].median(dim=['chain','draw']).values[-show_n_tr:]), color="black", linewidth=0.5)
    
    plt.fill_between(dates_train[-show_n_tr:],
                     np.expm1(train.posterior_predictive['obs'].quantile(q=0.025, dim=['chain','draw']).values[-show_n_tr:]),
                     np.expm1(train.posterior_predictive['obs'].quantile(q=0.975, dim=['chain','draw']).values[-show_n_tr:]),
                     color="black", alpha=0.05)
    plt.fill_between(dates_train[-show_n_tr:],
                     np.expm1(train.posterior_predictive['obs'].quantile(q=0.25, dim=['chain','draw']).values[-show_n_tr:]),
                     np.expm1(train.posterior_predictive['obs'].quantile(q=0.75, dim=['chain','draw']).values[-show_n_tr:]),
                     color="black", alpha=0.15)

    # Data Gap Connection
    gap_dates = (y_df['dt'] >= dates_train[-1]) & (y_df['dt'] <= dates_test[0])
    plt.plot(y_df.loc[gap_dates, 'dt'], np.expm1(y_df.loc[gap_dates, 'log_y_obs']), color="blue", linewidth=2.5)

    # Out-of-Sample Test Evaluation
    plt.plot(dates_test, np.expm1(y_df_test['log_y_obs'].values), linestyle=':', color="black", linewidth=2.5)
    plt.plot(dates_test, np.expm1(forecast.posterior_predictive['obs'].median(dim=['chain','draw'])), color="red", linewidth=1.5)
    
    plt.fill_between(dates_test,
                     np.expm1(forecast.posterior_predictive['obs'].quantile(q=0.025, dim=['chain','draw'])),
                     np.expm1(forecast.posterior_predictive['obs'].quantile(q=0.975, dim=['chain','draw'])),
                     color="red", alpha=0.05)
    plt.fill_between(dates_test,
                     np.expm1(forecast.posterior_predictive['obs'].quantile(q=0.25, dim=['chain','draw'])),
                     np.expm1(forecast.posterior_predictive['obs'].quantile(q=0.75, dim=['chain','draw'])),
                     color="red", alpha=0.15)

    # Baseline Model
    plt.plot(dates_test, null_forecast, color='green', linewidth=3)

    # Vertical Markers
    plt.axvline(x=dt_test[0], color='black', linestyle='--') 
    plt.axvline(x=dt_test[n_data_lag+1], color='black', linestyle='--') 

    title_str = f"Out-of-Sample Performance: Actual vs. Predicted Avoidable Deaths ({anchor_date.strftime('%Y-%m-%d')})"
    if null_score is not None:
        title_str += f" | Horizon null MSE: {null_score:.4f}"
    if model_score is not None:
        title_str += f" | Horizon model MSE: {model_score:.4f}"
        
    plt.title(title_str)
    plt.ylabel("Avoidable Deaths")
    plt.xlabel("Date")
    plt.grid(alpha=0.3)
    
    plt.gcf().autofmt_xdate()
    plt.tight_layout()
    
    file_name = f"goodness-of-fit_forecast_start_{anchor_date.strftime('%Y-%m-%d')}.pdf"
    plt.savefig(os.path.join(save_dir, file_name), bbox_inches='tight')
    plt.close()

In [99]:
def evaluate_forecasts(start_date, train_end, num_forecasts, horizon, train_trace):

    max_date = y_df['dt'].max()
    assert start_date >= train_end, "Evaluation start date must be after training window ends."
    assert (max_date - (start_date + timedelta(days=num_forecasts - 1))).days >= horizon, "Insufficient data for final horizon."   
    
    mse_records_1_5 = []
    mse_records_6_10 = []
    gam_mse_records_1_5 = []
    gam_mse_records_6_10 = []
    date_labels = []

    for step in range(num_forecasts):
        
        D_zero = start_date + timedelta(days=step)
        D_min3 = D_zero - timedelta(days=n_data_lag)
        D_plus10 = D_zero + timedelta(days=horizon)
        forecast_eval_dates = y_df[(y_df['dt'] > D_zero) & (y_df['dt'] <= D_plus10)]['dt'].values

        x_train_GAM = y_df[y_df['dt'] <= D_min3]['t']
        y_train_GAM = y_df[y_df['dt'] <= D_min3]['y_obs']   # NOT THE LOG
        x_predict_GAM = y_df[y_df['dt'] <= D_plus10]['t']
        gam = LinearGAM(s(0), fit_intercept=True).fit(x_train_GAM, y_train_GAM)
        gam_trend = pd.DataFrame(index=y_df[y_df['dt'] <= D_plus10]['dt'], data=gam.predict(x_predict_GAM), columns=['gam_trend',])

        flat_value = gam_trend.loc[D_min3, "gam_trend"]
        gam_trend.loc[D_min3:, "gam_trend"] = flat_value 

        gam_trend = gam_trend.reset_index()
        y_pred_gam = gam_trend[gam_trend['dt'].isin(forecast_eval_dates)]['gam_trend'].values
        y_vis_gam = gam_trend[((gam_trend['dt'] >= D_min3) & (gam_trend['dt'] <= D_plus10))]['gam_trend'].values

        y_slice = y_df[(y_df['dt'] >= D_min3) & (y_df['dt'] <= D_plus10)]
        X_slice = X_df[(X_df['dt'] >= D_min3) & (X_df['dt'] <= D_plus10)] 

        t_val = y_slice['t'].values
        dt_val = y_slice['dt'].values
        d_val = y_slice['day'].values
        y_val = y_slice['log_y_obs'].values
        y_val = np.concatenate((np.array([y_slice.iloc[0]['log_y_obs']]), np.full(len(y_val)-1, np.nan)), axis=0)   # retain only D-3 (other values NaN)
        y_actual = y_df[y_df['dt'].isin(forecast_eval_dates)]['y_obs'].values # NOT THE LOG

        unique_lags = sorted(X_slice['lag'].unique(), reverse=True)
        n_times = len(sorted(X_slice['t'].unique()))
        n_lags = len(unique_lags) 
        n_vars = len(X_slice['covariate_name'].unique())
        X_matrix = X_slice['covariate_value'].values.reshape(n_times, n_lags, n_vars)
        X_test = np.copy(X_matrix)

        for step_index in range(n_data_lag + 1, n_times):
            horizon_distance = step_index - n_data_lag - 1
            masked_lags = np.arange(n_lags) > horizon_distance
            X_test[step_index, ~masked_lags, :] = 0.0

        t_val = np.asarray(t_val, dtype=np.int64)
        d_val = np.asarray(d_val, dtype=np.int64)
        y_val = np.asarray(y_val, dtype=np.float64)
        X_test = np.asarray(X_test, dtype=np.float64)
t
        if hasattr(t_val, 'filled'): t_val = t_val.filled(0)
        if hasattr(d_val, 'filled'): d_val = d_val.filled(0)
        if hasattr(y_val, 'filled'): y_val = y_val.filled(0.0)
        if hasattr(X_test, 'filled'): X_test = X_test.filled(0.0)

        t_val = np.nan_to_num(t_val, nan=0).astype(np.int64)
        d_val = np.nan_to_num(d_val, nan=0).astype(np.int64)
        y_val = np.nan_to_num(y_val, nan=0.0)
        X_test = np.nan_to_num(X_test, nan=0.0)
        
        # Forecast
        with nhs_model:
            pm.set_data({"t_shared": t_val, "d_shared": d_val, "y_shared": y_val, "X_shared": X_test}, coords={"date": dt_val})
            forecast = pm.sample_posterior_predictive(trace, sample_vars=["mu_AR", "obs"], progressbar=False)

        # Compute forecast MSE
        y_pred = np.expm1(forecast["posterior_predictive/obs"].median(dim=['chain', 'draw']).sel(date=forecast_eval_dates)).values
        
        mse_model_1_5 = (1 / 5) * np.sum((y_actual[0:5] - y_pred[0:5])**2)
        mse_model_6_10 = (1 / 5) * np.sum((y_actual[5:10] - y_pred[5:10])**2)
        mse_records_1_5.append(mse_model_1_5)
        mse_records_6_10.append(mse_model_6_10)

        # Compute null model MSE
        mse_null_1_5 = (1 / 5) * np.sum((y_actual[0:5] - y_pred_gam[0:5])**2)
        mse_null_6_10 = (1 / 5) * np.sum((y_actual[5:10] - y_pred_gam[5:10])**2)
        gam_mse_records_1_5.append(mse_null_1_5)
        gam_mse_records_6_10.append(mse_null_6_10)

        date_labels.append(D_zero.strftime('%B %d, %Y'))
        
        mse_model_total = (1 / len(y_actual)) * np.sum((y_actual - y_pred)**2)
        mse_null_total = (1 / len(y_actual)) * np.sum((y_actual - y_pred_gam)**2)
        
        plot_window_trajectory(
            train=train_trace,
            forecast=forecast,
            y_df_test=y_slice,
            last_training_data_date=train_end,
            dt_test=dt_val,
            anchor_date=D_zero,
            null_forecast=y_vis_gam,
            null_score=mse_null_total,
            model_score=mse_model_total
        )

    metrics_df = pd.DataFrame({
        "Evaluation_Date": date_labels, 
        "MSE_model_1_5": mse_records_1_5, 
        "MSE_model_6_10": mse_records_6_10, 
        "MSE_null_1_5": gam_mse_records_1_5,
        "MSE_null_6_10": gam_mse_records_6_10
    })
    csv_out = os.path.join(project_root, "output", "backtest_mse_metrics.csv")
    os.makedirs(os.path.dirname(csv_out), exist_ok=True)
    metrics_df.to_csv(csv_out, index=False)
    
    print(f'Mean MSE for Bayesian model (Days 1-5): {np.mean(mse_records_1_5)}')
    print(f'Mean MSE for null model (Days 1-5): {np.mean(gam_mse_records_1_5)}')
    print(f'MSE ratio (Bayesian/null) (Days 1-5): {np.mean(mse_records_1_5)/np.mean(gam_mse_records_1_5)}')
    print(f'Mean MSE for Bayesian model (Days 6-10): {np.mean(mse_records_6_10)}')
    print(f'Mean MSE for null model (Days 6-10): {np.mean(gam_mse_records_6_10)}')
    print(f'MSE ratio (Bayesian/null) (Days 6-10): {np.mean(mse_records_6_10)/np.mean(gam_mse_records_6_10)}')
    
    return metrics_df

In [101]:
evaluate_forecasts(datetime(2025, 8, 16), training_end_date, 36, n_test, train)

Sampling: [mu_AR, obs]
Sampling: [mu_AR, obs]
Sampling: [mu_AR, obs]
Sampling: [mu_AR, obs]
Sampling: [mu_AR, obs]
Sampling: [mu_AR, obs]
Sampling: [mu_AR, obs]
Sampling: [mu_AR, obs]
Sampling: [mu_AR, obs]
Sampling: [mu_AR, obs]
Sampling: [mu_AR, obs]
Sampling: [mu_AR, obs]
Sampling: [mu_AR, obs]
Sampling: [mu_AR, obs]
Sampling: [mu_AR, obs]
Sampling: [mu_AR, obs]
Sampling: [mu_AR, obs]
Sampling: [mu_AR, obs]
Sampling: [mu_AR, obs]
Sampling: [mu_AR, obs]
Sampling: [mu_AR, obs]
Sampling: [mu_AR, obs]
Sampling: [mu_AR, obs]
Sampling: [mu_AR, obs]
Sampling: [mu_AR, obs]
Sampling: [mu_AR, obs]
Sampling: [mu_AR, obs]
Sampling: [mu_AR, obs]
Sampling: [mu_AR, obs]
Sampling: [mu_AR, obs]
Sampling: [mu_AR, obs]
Sampling: [mu_AR, obs]
Sampling: [mu_AR, obs]
Sampling: [mu_AR, obs]
Sampling: [mu_AR, obs]
Sampling: [mu_AR, obs]


Mean MSE for Bayesian model (Days 1-5): 0.035878736862244354
Mean MSE for null model (Days 1-5): 0.06363897941009213
MSE ratio (Bayesian/null) (Days 1-5): 0.5637855477071739
Mean MSE for Bayesian model (Days 6-10): 0.048599481917477386
Mean MSE for null model (Days 6-10): 0.07533272033877451
MSE ratio (Bayesian/null) (Days 6-10): 0.6451311156549692


,Evaluation_Date,MSE_model_1_5,MSE_model_6_10,MSE_null_1_5,MSE_null_6_10
0,"August 16, 2025",0.009099,0.055653,0.019793,0.100445
1,"August 17, 2025",0.013606,0.065361,0.016353,0.134657
2,"August 18, 2025",0.009571,0.153202,0.016240,0.249389
3,"August 19, 2025",0.022580,0.157273,0.012844,0.313861
4,"August 20, 2025",0.025223,0.157703,0.034260,0.299302
5,"August 21, 2025",0.052531,0.123280,0.099487,0.236711
6,"August 22, 2025",0.065242,0.100139,0.136433,0.196850
7,"August 23, 2025",0.132549,0.024669,0.245074,0.082322
8,"August 24, 2025",0.134580,0.005526,0.285437,0.029178
9,"August 25, 2025",0.133824,0.006705,0.272768,0.030336
